# Revolut FAQ RAG Chatbot — Build & Evaluate (Baseline)

A minimal single-turn RAG chatbot over Revolut help articles, evaluated
end-to-end on synthetic data with binary LLM-as-a-judge metrics.

**Stack:** `openai` for embeddings + chat, `numpy` for similarity search,
`json`/`pandas` for data. No frameworks, no vector DB — everything in memory.

**Pipeline:** generate synthetic queries → run them through RAG →
judge every answer → compute pass rates → iterate.

In [ ]:
%pip install -q openai numpy pandas python-dotenv tqdm seaborn matplotlib

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("BASE_URL"),
)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-3.5-turbo"  # weak model on purpose: makes failures easier to find and judges useful
TOP_K = 4

## 1. Load articles

In [ ]:
ARTICLES_PATH = "data/revolut_help_articles.jsonl"

articles = []
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        articles.append(json.loads(line))

print(f"Loaded {len(articles)} articles")
print("Example:", articles[0]["title"])

## 2. Embed all articles

We embed `title + content_text` so the title contributes to retrieval. Batched to keep things fast.

In [4]:
def article_to_text(a):
    return f"{a['title']}\n\n{a['content_text']}"

def embed_texts(texts, batch_size=100):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out, dtype=np.float32)

texts = [article_to_text(a) for a in articles]
embeddings = embed_texts(texts)

# L2-normalize once so cosine similarity is just a dot product
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (786, 1536)


## 3. Retrieval

In [5]:
def retrieve(query, k=TOP_K):
    q_emb = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), articles[i]) for i in top_idx]

## 4. Single-turn chat

In [6]:
SYSTEM_PROMPT = (
    "You are a Revolut customer support assistant. "
    "Answer the user's question using ONLY the provided help articles. "
    "If the answer is not in the articles, say you don't know. "
    "Be concise and reference steps from the articles when relevant."
)

def format_context(hits):
    parts = []
    for rank, (idx, score, art) in enumerate(hits, start=1):
        parts.append(
            f"[Article {rank}] {art['title']}\n{art['content_text']}"
        )
    return "\n\n---\n\n".join(parts)

def ask(question, k=TOP_K):
    hits = retrieve(question, k=k)
    context = format_context(hits)

    user_msg = (
        f"Help articles:\n\n{context}\n\n"
        f"Question: {question}"
    )

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )
    answer = resp.choices[0].message.content
    return answer, hits

## 5. Try it

In [8]:
question = "как открыть аккаунт в монзо"

answer, hits = ask(question)

print("Q:", question)
print("\nA:", answer)
print("\nRetrieved articles:")
for rank, (idx, score, art) in enumerate(hits, start=1):
    print(f"  {rank}. [{score:.3f}] {art['title']}")

Q: как открыть аккаунт в монзо

A: I'm sorry, I don't have information on how to open an account with Monzo.

Retrieved articles:
  1. [0.369] Open a Revolut – Kids & Teens account
  2. [0.357] Duplicate account
  3. [0.337] Open an investment account
  4. [0.331] Change or verify email address


## 6. Generate synthetic dataset

With zero real user data, we generate queries as the Cartesian product of persona × scenario × modifier. Personas define who is asking (country, tone, phrasing), scenarios define the support problem, and modifiers define the mood/circumstance. This product gives systematic coverage instead of ad-hoc examples.

In [ ]:
from generate_dataset import PERSONAS, SCENARIOS, MODIFIERS, generate_queries
import asyncio

# Show our seeds
print(f"Personas: {len(PERSONAS)}")
print(f"Scenarios: {len(SCENARIOS)}")
print(f"Modifiers: {len(MODIFIERS)}")
print(f"Total combinations: {len(PERSONAS) * len(SCENARIOS) * len(MODIFIERS)}")

# Generate or load queries
QUERIES_PATH = "data/synthetic_queries.csv"

if not os.path.exists(QUERIES_PATH):
    print("Generating synthetic queries...")
    asyncio.run(generate_queries(output_path=QUERIES_PATH, max_rows=None))

queries_df = pd.read_csv(QUERIES_PATH)
print(f"\nLoaded {len(queries_df)} synthetic queries")
queries_df.head()

## 7. Run synthetic queries through RAG

We run all synthetic queries through the RAG pipeline with async concurrency. Results are saved incrementally so we can resume if interrupted. Each output includes the query, generated answer, extracted context, and retrieved articles with scores.

In [ ]:
import asyncio
from tqdm import tqdm

RAG_OUTPUT_PATH = "data/synthetic_rag_outputs.csv"
RAG_CONCURRENCY = 8
SAVE_EVERY = 25
MAX_EVAL_ROWS = None  # Set to small int (e.g., 25) for testing, None for full run

async def run_rag_for_row(row, semaphore):
    """Run RAG for a single query row."""
    async with semaphore:
        query = row["query"]
        try:
            answer, hits = ask(query)
            
            # Format output
            retrieved_articles = []
            extracted_context = format_context(hits)
            
            for rank, (idx, score, art) in enumerate(hits, start=1):
                retrieved_articles.append({
                    "rank": rank,
                    "score": score,
                    "title": art["title"],
                })
            
            return {
                "persona": row["persona"],
                "scenario": row["scenario"],
                "modifier": row["modifier"],
                "query": query,
                "answer": answer,
                "extracted_context": extracted_context,
                "retrieved_articles": str(retrieved_articles),
            }
        except Exception as e:
            print(f"Error processing query: {e}")
            return None

async def run_all_rag(df, output_path, max_rows=None, concurrency=8, save_every=25):
    """Run RAG for all queries with incremental save."""
    if max_rows:
        df = df.head(max_rows)
    
    # Check existing
    existing = set()
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        for _, row in existing_df.iterrows():
            key = (row["persona"], row["scenario"], row["modifier"], row["query"])
            existing.add(key)
        print(f"Resuming: {len(existing)} already done")
    
    todo = []
    for _, row in df.iterrows():
        key = (row["persona"], row["scenario"], row["modifier"], row["query"])
        if key not in existing:
            todo.append(row)
    
    print(f"Remaining to run: {len(todo)}")
    
    if not todo:
        print("All queries already processed!")
        return pd.read_csv(output_path)
    
    # Setup CSV
    file_exists = os.path.exists(output_path)
    with open(output_path, "a", encoding="utf-8", newline="") as f:
        if not file_exists:
            f.write("persona,scenario,modifier,query,answer,extracted_context,retrieved_articles\n")
    
    semaphore = asyncio.Semaphore(concurrency)
    results = []
    
    for i, row in enumerate(tqdm(todo, desc="Running RAG")):
        try:
            result = await run_rag_for_row(row, semaphore)
            if result:
                # Escape quotes for CSV
                result_escaped = {k: v.replace('"', '""') if isinstance(v, str) else v 
                                   for k, v in result.items()}
                
                with open(output_path, "a", encoding="utf-8", newline="") as f:
                    f.write(f'"{result_escaped["persona"]}","{result_escaped["scenario"]}",'
                           f'"{result_escaped["modifier"]}","{result_escaped["query"]}",'
                           f'"{result_escaped["answer"]}","{result_escaped["extracted_context"]}",'
                           f'"{result_escaped["retrieved_articles"]}"\n')
                results.append(result)
                
                if (i + 1) % save_every == 0:
                    pass  # Already writing incrementally
        except Exception as e:
            print(f"Error: {e}")
            continue
    
    print(f"\nSaved {len(results)} results to {output_path}")
    return pd.read_csv(output_path)

# Run RAG
rag_results_df = await run_all_rag(
    queries_df, 
    output_path=RAG_OUTPUT_PATH,
    max_rows=MAX_EVAL_ROWS,
    concurrency=RAG_CONCURRENCY,
    save_every=SAVE_EVERY
)

print(f"\nTotal RAG outputs: {len(rag_results_df)}")
rag_results_df.head()

## 8. LLM-as-a-Judge evaluation

Each answer is judged by six binary criteria: relevancy (on-topic), not excessive (concise), helpful (actionable), no false information (grounded in context), legally correct (no financial advice/toxic content), and redirects when unknown. Each judge returns a boolean verdict and short reasoning.

In [ ]:
from judges import (
    judge_relevancy,
    judge_not_excessive,
    judge_is_helpful,
    judge_no_false_info,
    judge_legally_correct,
    judge_redirects_when_unknown,
)

EVAL_RESULTS_PATH = "data/synthetic_eval_results.csv"
JUDGE_CONCURRENCY = 8
MAX_EVAL_ROWS = None  # Set to small int for testing, None for full run

async def judge_row(row, semaphore):
    """Run all judges for a single RAG output."""
    async with semaphore:
        try:
            # Run all judges concurrently
            relevancy, rel_reasoning = await judge_relevancy(
                row["query"], row["answer"], row["extracted_context"]
            )
            not_excessive, exc_reasoning = await judge_not_excessive(
                row["query"], row["answer"], row["extracted_context"]
            )
            helpful, help_reasoning = await judge_is_helpful(
                row["query"], row["answer"], row["extracted_context"]
            )
            no_false_info, false_info_reasoning = await judge_no_false_info(
                row["query"], row["answer"], row["extracted_context"]
            )
            legally_correct, legal_reasoning = await judge_legally_correct(
                row["query"], row["answer"], row["extracted_context"]
            )
            redirects_ok, redirect_reasoning = await judge_redirects_when_unknown(
                row["query"], row["answer"], row["extracted_context"]
            )

            return {
                "persona": row["persona"],
                "scenario": row["scenario"],
                "modifier": row["modifier"],
                "query": row["query"],
                "answer": row["answer"],
                "judge_relevancy": relevancy,
                "judge_relevancy_reasoning": rel_reasoning,
                "judge_not_excessive": not_excessive,
                "judge_not_excessive_reasoning": exc_reasoning,
                "judge_is_helpful": helpful,
                "judge_is_helpful_reasoning": help_reasoning,
                "judge_no_false_info": no_false_info,
                "judge_no_false_info_reasoning": false_info_reasoning,
                "judge_legally_correct": legally_correct,
                "judge_legally_correct_reasoning": legal_reasoning,
                "judge_redirects_when_unknown": redirects_ok,
                "judge_redirects_when_unknown_reasoning": redirect_reasoning,
            }
        except Exception as e:
            print(f"Error judging row: {e}")
            return None

async def run_all_judges(df, output_path, max_rows=None, concurrency=8):
    """Run judges for all RAG outputs with incremental save."""
    if max_rows:
        df = df.head(max_rows)
    
    # Check existing
    existing = set()
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        for _, row in existing_df.iterrows():
            key = (row["persona"], row["scenario"], row["modifier"], row["query"])
            existing.add(key)
        print(f"Resuming: {len(existing)} already judged")
    
    todo = []
    for _, row in df.iterrows():
        key = (row["persona"], row["scenario"], row["modifier"], row["query"])
        if key not in existing:
            todo.append(row)
    
    print(f"Remaining to judge: {len(todo)}")
    
    if not todo:
        print("All queries already judged!")
        return pd.read_csv(output_path)
    
    # Setup CSV
    file_exists = os.path.exists(output_path)
    with open(output_path, "a", encoding="utf-8", newline="") as f:
        if not file_exists:
            f.write("persona,scenario,modifier,query,answer,judge_relevancy,judge_relevancy_reasoning,judge_not_excessive,judge_not_excessive_reasoning,judge_is_helpful,judge_is_helpful_reasoning,judge_no_false_info,judge_no_false_info_reasoning,judge_legally_correct,judge_legally_correct_reasoning,judge_redirects_when_unknown,judge_redirects_when_unknown_reasoning\n")
    
    semaphore = asyncio.Semaphore(concurrency)
    
    for i, row in enumerate(tqdm(todo, desc="Judging")):
        try:
            result = await judge_row(row, semaphore)
            if result:
                # Escape values for CSV
                result_escaped = {}
                for k, v in result.items():
                    if isinstance(v, str):
                        v_escaped = v.replace('"', '""')
                        result_escaped[k] = f'"{v_escaped}"'
                    elif isinstance(v, bool):
                        result_escaped[k] = str(v)
                    else:
                        result_escaped[k] = str(v)
                
                line = ",".join([result_escaped[k] for k in result.keys()])
                with open(output_path, "a", encoding="utf-8", newline="") as f:
                    f.write(line + "\n")
        except Exception as e:
            print(f"Error: {e}")
            continue
    
    print(f"\nSaved evaluation results to {output_path}")
    return pd.read_csv(output_path)

# Run judges
eval_results_df = await run_all_judges(
    rag_results_df,
    output_path=EVAL_RESULTS_PATH,
    max_rows=MAX_EVAL_ROWS,
    concurrency=JUDGE_CONCURRENCY,
)

print(f"\nTotal evaluated: {len(eval_results_df)}")
eval_results_df.head()

## 9. Results: pass rates

We compute the mean pass rate per judge and visualize as a horizontal bar chart. This shows which quality dimensions fail most often and where prompt optimization (stage 02) should focus.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compute pass rates per judge
judge_cols = [
    "judge_relevancy",
    "judge_not_excessive",
    "judge_is_helpful",
    "judge_no_false_info",
    "judge_legally_correct",
    "judge_redirects_when_unknown",
]

pass_rates = {}
for col in judge_cols:
    pass_rate = eval_results_df[col].mean()
    judge_name = col.replace("judge_", "").replace("_", " ").title()
    pass_rates[judge_name] = pass_rate

# Create DataFrame for display
pass_rates_df = pd.DataFrame.from_dict(pass_rates, orient="index", columns=["Pass Rate"])
pass_rates_df = pass_rates_df.sort_values("Pass Rate", ascending=True)

print("Overall LLM-as-a-judge pass rates (baseline):")
pass_rates_df

# Plot
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    x="Pass Rate",
    y=pass_rates_df.index,
    data=pass_rates_df.reset_index(),
    palette="RdYlGn",
    hue="index",
    legend=False,
)
ax.set_xlim(0, 1)
plt.title("Overall LLM-as-a-judge pass rates (baseline)", fontsize=14)
plt.xlabel("Pass Rate", fontsize=12)
plt.ylabel("Judge", fontsize=12)
plt.tight_layout()
plt.show()

## Takeaways

The pass rates show which quality dimensions are failing most. The weak chat model (gpt-3.5-turbo class) makes this visible — helpfulness and conciseness are likely failure points, while legal correctness should be near-perfect (the task is simple safety). Relevancy failures indicate retrieval issues, while "no false information" failures indicate hallucination or going beyond context. This frozen eval set (synthetic_queries.csv) becomes the baseline for stage 02 (GEPA optimization), where we'll systematically improve prompts and re-run the SAME evaluation to measure progress.